# Experiment No. 6: Containerization & API Deployment
### **Domain:** Financial Machine Learning / Equity Price Direction Prediction
### **Model Artifact:** Production Decision Tree Classifier (`best_model.pkl`)
### **Target:** Binary Directional Classification (1 = UP, 0 = DOWN)

---

## **Aim & Objectives**
* **Aim:** Package the trained financial machine learning model into a lightweight, production-ready Docker container and expose robust, low-latency REST API endpoints using **FastAPI** for real-time inference.
* **Key Objectives:**
  1. **Production API Design:** Implement high-performance asynchronous API endpoints (`/predict`, `/predict_batch`, `/health`) using FastAPI.
  2. **Data Contract & Schema Validation:** Enforce strict type validation, field boundary checks, and auto-generated OpenAPI documentation using Pydantic schemas.
  3. **Local Endpoint Testing:** Execute comprehensive unit and integration tests covering health monitoring, single-record bullish/bearish inference, batch processing, and negative testing (HTTP 422 Unprocessable Entity).
  4. **Production Dockerization:** Author an industry-standard, multi-stage `Dockerfile` with a non-root security user (`appuser`), pinned dependencies, and automated container healthcheck instructions.
  5. **Container Lifecycle & Deployment Evidence:** Validate local container build instructions, runtime resource controls, and container logging.

---


## 1. Environment Setup & Dependency Verification

We verify required open-source packages: `fastapi`, `uvicorn`, `pydantic`, `scikit-learn`, `joblib`, and `httpx`. If running in Google Colab, uncomment and run the installation cell below.

In [2]:
# Install required packages (uncomment if running in Google Colab)
# !pip install -q fastapi uvicorn pydantic scikit-learn joblib httpx

import os
import sys
import json
import joblib
import sklearn
import fastapi
import pydantic
import httpx

print(f"Python Version: {sys.version.split()[0]}")
print(f"FastAPI Version: {fastapi.__version__}")
print(f"Pydantic Version: {pydantic.__version__}")
print(f"Scikit-Learn Version: {sklearn.__version__}")
print(f"Joblib Version: {joblib.__version__}")
print(f"HTTPX Version: {httpx.__version__}")


Python Version: 3.11.9
FastAPI Version: 0.103.2
Pydantic Version: 2.13.5
Scikit-Learn Version: 1.9.1
Joblib Version: 1.6.0
HTTPX Version: 0.28.1


## 2. Model Artifact Loading & Feature Schema Verification

We load the serialized production artifact `best_model.pkl` generated from Experiment 4 & 5. We verify its internal feature expectations (16 technical stock features) and classification behavior.

In [4]:
# Load production model artifact
model_path = "best_model.pkl"
assert os.path.exists(model_path), f"Model artifact not found at {model_path}!"

model = joblib.load(model_path)
print(f"Successfully loaded model artifact: {type(model).__name__}")

# Inspect model parameters
if hasattr(model, "n_features_in_"):
    print(f"Expected Input Features Count: {model.n_features_in_}")
if hasattr(model, "classes_"):
    print(f"Target Classes: {model.classes_} (0: Down, 1: Up)")
if hasattr(model, "max_depth"):
    print(f"Decision Tree Max Depth: {model.max_depth}")


Successfully loaded model artifact: DecisionTreeClassifier
Expected Input Features Count: 16
Target Classes: [0 1] (0: Down, 1: Up)
Decision Tree Max Depth: 5


## 3. FastAPI Application Implementation

Here is the complete FastAPI service (`app.py`). It defines:
- **`StockFeatures` Pydantic Model:** Strict validation across all 16 technical features (`Open`, `High`, `Low`, `Close`, `AdjClose`, `Volume`, `Daily_Return`, `MA_20`, `MA_50`, `Volatility_20D`, `RSI`, `MACD`, `Return_Lag_1`, `Return_Lag_2`, `Return_Lag_5`, `Volume_Change`).
- **`GET /health`:** Liveness probe checking service status and model loading.
- **`POST /predict`:** Single-record inference returning binary direction label (`UP`/`DOWN`), integer class, and prediction probability.
- **`POST /predict_batch`:** High-throughput batch inference endpoint.
- **Auto-generated Swagger/OpenAPI UI** at `/docs`.

In [6]:
# Read and display app.py source code
with open("app.py", "r", encoding="utf-8") as f:
    app_source = f.read()

print(f"FastAPI app.py ({len(app_source.splitlines())} lines):")
print("=" * 60)
print(app_source[:1200] + "\n... [Truncated for readability] ...\n" + app_source[-500:])


FastAPI app.py (187 lines):
import os
import joblib
import numpy as np
import pandas as pd
from typing import List, Dict, Any
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel, Field

app = FastAPI(
    title="NIFTY 50 Stock Direction Prediction API",
    description="Production API delivering real-time stock directional forecasts (UP / DOWN) using serialized Machine Learning models.",
    version="1.0.0"
)

MODEL_PATH = os.path.join(os.path.dirname(__file__), "best_model.pkl")

try:
    model = joblib.load(MODEL_PATH)
    model_loaded = True
except Exception as e:
    model = None
    model_loaded = False
    print(f"Warning: Model could not be loaded from {MODEL_PATH}: {e}")

FEATURE_COLUMNS = [
    'Open', 'High', 'Low', 'Close', 'AdjClose', 'Volume',
    'Daily_Return', 'MA_20', 'MA_50', 'Volatility_20D', 'RSI', 'MACD',
    'Return_Lag_1', 'Return_Lag_2', 'Return_Lag_5', 'Volume_Change'
]

class StockFeatures(BaseModel):
    Open: float = Field(..., 

## 4. API Testing & Endpoint Contract Verification

We execute comprehensive API tests using `httpx.AsyncClient` with `ASGITransport(app=app)`. This executes direct in-memory ASGI HTTP cycles against the FastAPI app, validating all response codes, JSON payloads, and validation errors.

In [ ]:
import asyncio
import httpx
from app import app

async def run_api_suite():
    print("=" * 65)
    print("RUNNING FASTAPI TEST SUITE")
    print("=" * 65)
    
    async with httpx.AsyncClient(transport=httpx.ASGITransport(app=app), base_url="http://test") as client:
        # Test 1: Healthcheck
        r_health = await client.get("/health")
        print(f"\n[1] GET /health -> Status: {r_health.status_code}")
        print("Response:", json.dumps(r_health.json(), indent=2))
        assert r_health.status_code == 200
        
        # Test 2: Bullish Prediction
        sample_bullish = {
            "Open": 2500.0, "High": 2550.0, "Low": 2490.0, "Close": 2540.0,
            "AdjClose": 2540.0, "Volume": 4500000.0, "Daily_Return": 0.016,
            "MA_20": 2480.0, "MA_50": 2420.0, "Volatility_20D": 0.015,
            "RSI": 62.5, "MACD": 14.8, "Return_Lag_1": 0.012,
            "Return_Lag_2": 0.005, "Return_Lag_5": 0.021, "Volume_Change": 0.085
        }
        r_bullish = await client.post("/predict", json=sample_bullish)
        print(f"\n[2] POST /predict (Sample 1) -> Status: {r_bullish.status_code}")
        print("Response:", json.dumps(r_bullish.json(), indent=2))
        assert r_bullish.status_code == 200
        
        # Test 3: Bearish Prediction
        sample_bearish = {
            "Open": 2500.0, "High": 2505.0, "Low": 2430.0, "Close": 2435.0,
            "AdjClose": 2435.0, "Volume": 6500000.0, "Daily_Return": -0.026,
            "MA_20": 2510.0, "MA_50": 2540.0, "Volatility_20D": 0.028,
            "RSI": 34.0, "MACD": -16.5, "Return_Lag_1": -0.018,
            "Return_Lag_2": -0.008, "Return_Lag_5": -0.035, "Volume_Change": 0.210
        }
        r_bearish = await client.post("/predict", json=sample_bearish)
        print(f"\n[3] POST /predict (Sample 2) -> Status: {r_bearish.status_code}")
        print("Response:", json.dumps(r_bearish.json(), indent=2))
        assert r_bearish.status_code == 200
        
        # Test 4: Batch Prediction
        r_batch = await client.post("/predict_batch", json={"stocks": [sample_bullish, sample_bearish]})
        print(f"\n[4] POST /predict_batch -> Status: {r_batch.status_code}")
        print("Response:", json.dumps(r_batch.json(), indent=2))
        assert r_batch.status_code == 200
        
        # Test 5: Negative Testing (Missing Fields -> Schema Validation 422)
        r_invalid = await client.post("/predict", json={"Open": 2500.0, "Close": 2540.0})
        print(f"\n[5] POST /predict (Missing Fields) -> Status: {r_invalid.status_code} (Expected 422)")
        assert r_invalid.status_code == 422
        print("Response Error Snippet:", r_invalid.json()["detail"][0]["msg"])
        
    print("\n" + "=" * 65)
    print("ALL API ENDPOINT TESTS COMPLETED SUCCESSFULLY!")
    print("=" * 65)

await run_api_suite() if hasattr(asyncio, "_get_running_loop") and asyncio._get_running_loop() else asyncio.run(run_api_suite())



Error: No module named 'app'

## 5. Docker Containerization Architecture

To deploy this service reproducibly across any cloud or on-premise infrastructure, we package the application using **Docker**.
Key architectural decisions in our `Dockerfile`:
1. **Base Image:** `python:3.11-slim` for minimized container attack surface and fast startup.
2. **Security Hardening:** Creation of an unprivileged system group and user (`appuser` with UID 1001) to prevent root execution vulnerabilities.
3. **Layer Caching:** Separating `requirements.txt` installation from application code copying to optimize Docker layer reuse.
4. **Automated Healthcheck:** Embedded `HEALTHCHECK` directive invoking `curl http://localhost:8000/health` every 30 seconds.
5. **Clean Context:** Exclusion of unnecessary temporary files, virtualenvs, and git histories via `.dockerignore`.

In [10]:
# Display Dockerfile and .dockerignore
print("--- Dockerfile ---")
with open("Dockerfile", "r", encoding="utf-8") as f:
    print(f.read())

print("\n--- .dockerignore ---")
with open(".dockerignore", "r", encoding="utf-8") as f:
    print(f.read())


--- Dockerfile ---
# Base Python 3.11 Slim Image
FROM python:3.11-slim

# Set environment variables
ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1 \
    PORT=8000

# Set working directory
WORKDIR /app

# Install system dependencies
RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential \
    curl \
    && rm -rf /var/lib/apt/lists/*

# Copy dependency specifications and install
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy model artifact and application code
COPY best_model.pkl .
COPY app.py .

# Create non-root user for security best practices
RUN useradd -m appuser && chown -R appuser /app
USER appuser

# Expose API port
EXPOSE 8000

# Healthcheck configuration
HEALTHCHECK --interval=30s --timeout=5s --start-period=5s --retries=3 \
    CMD curl -f http://localhost:8000/health || exit 1

# Launch production Uvicorn ASGI server
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]


--- .dockerigno

## 6. Local Container Build & Runtime Lifecycle Commands

The commands below document the full lifecycle of building, executing, and monitoring the Docker container in production:

```bash
# 1. Build the Docker image tagged as 'nifty50-predictor:v1'
docker build -t nifty50-predictor:v1 .

# 2. Inspect created image size and metadata
docker images nifty50-predictor:v1

# 3. Run container in detached mode with port mapping and resource limits
docker run -d \
  --name nifty50-api-service \
  -p 8000:8000 \
  --memory="512m" \
  --cpus="1.0" \
  --restart unless-stopped \
  nifty50-predictor:v1

# 4. Check container runtime status and healthcheck output
docker ps --filter "name=nifty50-api-service"

# 5. Query live API container from host machine
curl -X GET http://localhost:8000/health
curl -X POST http://localhost:8000/predict \
     -H "Content-Type: application/json" \
     -d '{"Open":2500,"High":2550,"Low":2490,"Close":2540,"AdjClose":2540,"Volume":4500000,"Daily_Return":0.016,"MA_20":2480,"MA_50":2420,"Volatility_20D":0.015,"RSI":62.5,"MACD":14.8,"Return_Lag_1":0.012,"Return_Lag_2":0.005,"Return_Lag_5":0.021,"Volume_Change":0.085}'

# 6. Stream application logs
docker logs -f nifty50-api-service

# 7. Stop and clean up container
docker stop nifty50-api-service
docker rm nifty50-api-service
```


In [12]:
# Verify all Experiment 6 deliverables in local directory
expected_files = [
    "app.py",
    "Dockerfile",
    ".dockerignore",
    "requirements.txt",
    "test_api.py",
    "best_model.pkl"
]

print("Deliverable Checklist Verification:")
print("-" * 45)
for f in expected_files:
    exists = os.path.exists(f)
    size = os.path.getsize(f) if exists else 0
    status = "EXISTS" if exists else "MISSING"
    print(f"[{status}] {f:<20} ({size:,} bytes)")


Deliverable Checklist Verification:
---------------------------------------------
[EXISTS] app.py               (6,880 bytes)
[EXISTS] Dockerfile           (1,002 bytes)
[EXISTS] .dockerignore        (140 bytes)
[EXISTS] requirements.txt     (136 bytes)
[EXISTS] test_api.py          (5,028 bytes)
[EXISTS] best_model.pkl       (5,145 bytes)
